# 02 Trening av dype nevrale nettverk
*Data Mining & Analytics – Samling 3, dag 1*


Her bruker vi Fashion-MNIST: 70 000 svartt/hvitt bilder (28 × 28) av klær i 10 klasser.

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 4.5)
rng = np.random.default_rng(42)
tf.keras.utils.set_random_seed(42)
print("TensorFlow", tf.__version__)

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
klasser = ["T-skjorte/topp", "Bukse", "Genser", "Kjole", "Kåpe",
           "Sandal", "Skjorte", "Joggesko", "Veske", "Ankelstøvel"]

In [ ]:
(X_full, y_full), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
X_full = (X_full / 255.0).astype("float32") # skaler pikselverdiene til [0, 1]
X_test = (X_test / 255.0).astype("float32")

# bruker bare noe av treningsdataene for å korte ned treningstiden
X_train, y_train = X_full[:20_000], y_full[:20_000]
X_val, y_val = X_full[-5_000:], y_full[-5_000:]
print("Trening:", X_train.shape, " validering:", X_val.shape, " test:", X_test.shape)

fig, ax = plt.subplots(2, 10, figsize=(16, 3.6))
for k in range(10):
    idx = np.where(y_train == k)[0][:2]
    for r in range(2):
        ax[r, k].imshow(X_train[idx[r]], cmap="gray_r")
        ax[r, k].axis("off")
    ax[0, k].set_title(klasser[k], fontsize=9)
plt.show()

## 1. Vanishing gradients (Géron s. 275–282)

Under backpropagation multipliseres gradienten med den deriverte av aktiveringsfunksjonen og vektene i hvert lag.
Den deriverte av sigmoid er aldri større enn 0,25, så gradienten krymper for hvert lag den sendes bakover.
Med mange lag blir gradientene i de første lagene forsvinnende små, slik at de første lagene lærer nesten ingenting.
Glorot og Bengio (2010) analyserte dette og foreslo en bedre initialisering; He et al. (2015) tilpasset den til ReLU,
som har derivert 1 for positive verdier.

Vi måler normen til gradienten for vektene i hvert lag i et nettverk med 10 skjulte lag, på én batch, før trening.

In [ ]:
def dypt_nett(aktivering, init, n_lag=10, batchnorm=False):
    lag = [tf.keras.Input(shape=(28, 28)), tf.keras.layers.Flatten()]
    for _ in range(n_lag):
        lag.append(tf.keras.layers.Dense(100, kernel_initializer=init))
        if batchnorm:
            lag.append(tf.keras.layers.BatchNormalization())
        lag.append(tf.keras.layers.Activation(aktivering))
    lag.append(tf.keras.layers.Dense(10, activation="softmax"))
    return tf.keras.Sequential(lag)

def gradientnorm_per_lag(modell, X, y):
    with tf.GradientTape() as tape:
        tap = tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(y, modell(X, training=True)))
    grads = tape.gradient(tap, modell.trainable_variables)
    kjerner = [g for g, v in zip(grads, modell.trainable_variables) if len(v.shape) == 2]   # kun vektmatriser
    return [float(tf.norm(g)) for g in kjerner[:-1]]                                        # skjulte lag

oppsett = {
    "Sigmoid + Glorot": ("sigmoid", "glorot_uniform"),
    "Tanh + Glorot":    ("tanh", "glorot_uniform"),
    "ReLU + He":        ("relu", "he_normal"),
}
for navn, (akt, init) in oppsett.items():
    normer = gradientnorm_per_lag(dypt_nett(akt, init), X_train[:256], y_train[:256])
    plt.semilogy(range(1, 11), normer, "o-", label=navn)
plt.xlabel("Skjult lag (1 = nærmest input)")
plt.ylabel("log-skala")
plt.title("Gradient-verdier per lag før trening")
plt.legend()
plt.show()

Med sigmoid-aktivering er gradienten i lag 1 mye mindre enn i lag 10. Med ReLU og He-initialisering er gradientene
omtrent like store gjennom hele nettverket. (Sammenlign med figuren i forelesningen.)

Batch-normalisering standardiserer input til hvert lag per mini-batch. Vi trener det dype sigmoid-nettverket med og uten batch-normalisering og sammenligner læringen:

In [ ]:
resultater = {}
for navn, bn in [("Sigmoid, 10 lag", False), ("Sigmoid, 10 lag + BatchNorm", True)]:
    m = dypt_nett("sigmoid", "glorot_uniform", batchnorm=bn)
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    h = m.fit(X_train, y_train, epochs=5, batch_size=64, validation_data=(X_val, y_val), verbose=0)
    resultater[navn] = h.history["val_accuracy"]

for navn, acc in resultater.items():
    plt.plot(range(1, 6), acc, "o-", label=navn)
plt.xlabel("Epoke"); plt.ylabel("Valideringsnøyaktighet"); plt.legend()
plt.title("Effekten av batch-normalisering i et dypt sigmoid-nettverk")
plt.show()

## 2. Raskere optimeringsalgoritmer (Géron s. 293–302)

| Algoritme | Idé |
|---|---|
| SGD | $\theta \leftarrow \theta - \eta \nabla J(\theta)$ |
| Momentum | Akkumulerer en «fart» i retningen gradientene peker over tid – ruller raskere ned bakker |
| RMSProp | Skalerer læringsraten per parameter med et glidende snitt av kvadrerte gradienter |
| Adam | Kombinerer momentum og RMSProp – et godt standardvalg |

In [ ]:
def lag_mlp(dropout=0.0, l2=0.0, bredde=(256, 128)):
    reg = tf.keras.regularizers.l2(l2) if l2 else None
    lag = [tf.keras.Input(shape=(28, 28)), tf.keras.layers.Flatten()]
    for n in bredde:
        lag.append(tf.keras.layers.Dense(n, activation="relu", kernel_initializer="he_normal", kernel_regularizer=reg))
        if dropout:
            lag.append(tf.keras.layers.Dropout(dropout))
    lag.append(tf.keras.layers.Dense(10, activation="softmax"))
    return tf.keras.Sequential(lag)

optimering = {
    "SGD":            lambda: tf.keras.optimizers.SGD(learning_rate=0.01),
    "SGD + momentum": lambda: tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    "RMSProp":        lambda: tf.keras.optimizers.RMSprop(learning_rate=0.001),
    "Adam":           lambda: tf.keras.optimizers.Adam(learning_rate=0.001),
}
for navn, opt in optimering.items():
    m = lag_mlp()
    m.compile(optimizer=opt(), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    h = m.fit(X_train, y_train, epochs=8, batch_size=64, validation_data=(X_val, y_val), verbose=0)
    plt.plot(range(1, 9), h.history["val_loss"], "o-", label=navn)
plt.xlabel("Epoke"); plt.ylabel("Valideringstap"); plt.legend()
plt.title("Sammenligning av optimeringsalgoritmer")
plt.show()

Læringsraten er kanskje den viktigste hyperparameteren. Er den for liten går treningen sakte, men er den for stor divergerer den.
Et alternativ er å starte høyt og senke læringsraten underveis, f.eks.
`tf.keras.optimizers.schedules.ExponentialDecay(initial_learning_rate=1e-3, decay_steps=1000, decay_rate=0.9)`.

## 3. Overtilpasning og regularisering (Géron s. 302–310)

Et dypt nettverk med hundretusener av parametere kan lett overtilpasse et lite treningssett. Vi trener et stort MLP på bare
3 000 bilder, med og uten regularisering:
* **Early stopping**: stopp når validerings-loss slutter å synke seg
* **L2-regularisering**: straffer store vekter (som Ridge i samling 2)
* **Dropout**: hvert nevron slås av med sannsynlighet *p* i hvert treningssteg, slik at nettverket ikke kan stole på enkeltnevroner

In [ ]:
X_lite, y_lite = X_train[:3_000], y_train[:3_000]

uten = lag_mlp(bredde=(512, 512))
uten.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h_uten = uten.fit(X_lite, y_lite, epochs=40, batch_size=64, validation_data=(X_val, y_val), verbose=0)

med = lag_mlp(bredde=(512, 512), dropout=0.4, l2=1e-4)
med.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
stopp = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
h_med = med.fit(X_lite, y_lite, epochs=40, batch_size=64, validation_data=(X_val, y_val), callbacks=[stopp], verbose=0)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for a, h, tittel in [(ax[0], h_uten, "Uten regularisering"), (ax[1], h_med, "Dropout + ℓ2 + early stopping")]:
    a.plot(h.history["loss"], label="trening")
    a.plot(h.history["val_loss"], label="validering")
    a.set_title(tittel); a.set_xlabel("Epoke"); a.legend()
ax[0].set_ylabel("Tap")
plt.show()

for navn, m in [("Uten regularisering", uten), ("Med regularisering", med)]:
    _, acc = m.evaluate(X_val, y_val, verbose=0)
    print(f"{navn:22s} valideringsnøyaktighet = {acc:.3f}")

Uten regularisering synker treningstapet mot null mens valideringstapet øker – klassisk overtilpasning.
Med dropout, L2 og early stopping følger kurvene hverandre tettere, og valideringsnøyaktigheten blir bedre.

## Praktiske retningslinjer

Géron (tabell 11-2, s. 310) foreslår følgende standardoppsett som et godt utgangspunkt:

| Valg | Standard |
|---|---|
| Initialisering | He-initialisering |
| Aktiveringsfunksjon | ReLU  |
| Normalisering | Batch-normalisering |
| Regularisering | Dropout (+ early stopping) |
| Optimalisering | Adam |

gjenbruk gjerne et forhåndstrent nettverk hvis det finnes ett for et lignende problem.